# 第2章 NumPy 配列 ―― たくさんの数をまとめて計算する

**この章でできるようになること**
- `np.array`・`np.linspace`・`np.arange` で配列を作れる
- 配列の形（`shape`）と番号（インデックス）、切り出し（スライス）を使える
- 2次元配列（研究の「IV マップ」の形）と `axis` を使える
- ブロードキャストで「電流 × 振幅」の表を `for` なしで作れる
- ブールマスク（True/False の配列）で条件に合う点だけ取り出せる

**例え：** Python のリストは「1つずつ手で運ぶバケツリレー」、NumPy 配列は「ベルトコンベア」です。同じ作業を全部の値にまとめて一度に流せるので、速くて書くのも短くなります。

| 用語 | 意味 | 例え |
|---|---|---|
| NumPy | 数値計算のためのライブラリ。配列を速く計算できる | 電卓をまとめて何百台も並べた工場 |
| 配列（ndarray） | 同じ型の数を、表の形に並べたもの | 方眼紙のマス目 |
| shape | 配列の各方向の大きさ。例：`(636, 33)` | 方眼紙の「縦◯マス×横◯マス」 |
| 軸（axis） | 2次元なら axis=0 が縦（行の方向）、axis=1 が横（列の方向） | 表を縦に足すか横に足すか |
| ベクトル化 | `for` で1個ずつではなく、配列全体にまとめて計算すること | ベルトコンベアで一斉に処理 |
| ブロードキャスト | 形の違う配列を、自動で引き伸ばしてそろえて計算すること | 1枚のはんこを、表の全部の行に押す |
| ブールマスク | True/False の配列。True の場所だけ取り出すのに使う | 穴の開いた型紙を重ねて、穴の部分だけ見る |
| ビュー（view） | 元の配列の一部を「のぞいている」だけのもの。書き換えると元も変わる | 窓から見た景色（窓を拭いても景色そのものは同じもの） |

> 研究コードとのつながり：`py/rcsj.py` の `iv_map` は形 `(M, nA)`（電流の数 × 振幅の数）の配列を返します。`py/load.py` の `extract` は `np.diff`・ブールマスク・`np.where` を使っています。

In [1]:
# 準備：教材の部品（kyozai/）を import できるようにする。どの章でも最初に1回だけ実行する。
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "kyozai").exists():   # kyozai/ があるフォルダまで上にたどる
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("教材の場所:", ROOT.name)

教材の場所: python-from-my-research


## 2.1 配列を作る

`import numpy as np` は「NumPy を `np` という短い名前で使う」という意味で、ほぼ決まり文句です。

In [2]:
import numpy as np

a = np.array([0.5, 0.8, 1.3])
print(a, a.shape, a.dtype)

A_list = np.linspace(0, 64, 33)         # 0 から 64 までを 33 点（両端を含む）
print(A_list[:5], "...", A_list[-1], "点の数:", A_list.size)

I = np.arange(1.6, 4.2, 0.0070779)      # 1.6 から 0.0070779 刻み（4.2 は含まない）
print(I[:3], "点の数:", I.size)

[0.5 0.8 1.3] (3,) float64
[0. 2. 4. 6. 8.] ... 64.0 点の数: 33
[1.6       1.6070779 1.6141558] 点の数: 368


- `np.linspace(始め, 終わり, 点の数)` … MATLAB の `linspace` と同じ。終わりを**含む**。
- `np.arange(始め, 終わり, 刻み)` … MATLAB の `始め:刻み:終わり` に近いが、終わりを**含まない**。

研究の保存データの電流の刻みは β=10 で 0.0070779 です（`lab/式と前提.md` 2章）。

## 2.2 配列どうしの計算は「要素ごと」

NumPy では `*` や `/` は**要素ごと**の計算です。MATLAB の `.*` `./` にあたります（行列の積は `@`）。

In [3]:
n = np.array([0, 1, -1, 1, 2])
m = np.array([1, 0, 1, 1, 0])
V = n * 0.5 + m * 0.8          # 5つのステップの電圧を一度に計算
print(V)
print(np.round(V, 3))

[0.8 0.5 0.3 1.3 1. ]
[0.8 0.5 0.3 1.3 1. ]


`for` で1つずつ計算した場合と比べてみます。`%timeit` は Jupyter の便利機能で、何回も実行して平均時間を測ります。

In [4]:
x = np.linspace(0, 100, 100_000)

def with_loop(x):
    out = []
    for v in x:
        out.append(np.sin(0.5 * v) + np.sin(0.8 * v))
    return out

%timeit -n 1 -r 3 with_loop(x)
%timeit -n 10 -r 3 np.sin(0.5 * x) + np.sin(0.8 * x)

93 ms ± 2.5 ms per loop (mean ± std. dev. of 3 runs, 1 loop each)
786 µs ± 57.4 µs per loop (mean ± std. dev. of 3 runs, 10 loops each)


配列にまとめて計算するほうが、ずっと速いはずです（何倍かは計算機によります）。

## 2.3 インデックスとスライス

In [5]:
t = np.arange(10)
print(t)
print(t[0], t[-1])      # 最初と最後
print(t[2:5])           # 2, 3, 4
print(t[::2])           # 1つおき
print(t[::-1])          # 逆順

[0 1 2 3 4 5 6 7 8 9]
0 9
[2 3 4]
[0 2 4 6 8]
[9 8 7 6 5 4 3 2 1 0]


**注意：スライスは「ビュー」**です。切り出したものを書き換えると、元の配列も変わります。独立したコピーがほしいときは `.copy()` を使います。MATLAB では代入するとコピーになるので、ここは大きな違いです。

In [6]:
t = np.arange(5)
part = t[1:3]
part[0] = 100
print("元の t:", t)          # t も変わっている！

t = np.arange(5)
part = t[1:3].copy()
part[0] = 100
print("copy したとき:", t)   # t は変わらない

元の t: [  0 100   2   3   4]
copy したとき: [0 1 2 3 4]


## 2.4 2次元配列と axis ―― IV マップの形

研究の `iv_map` は、**行 = 電流、列 = 振幅** の表 `V[i, a]` を返します。ここでは中身の代わりに簡単な式で同じ形の表を作ってみます。

In [7]:
I = np.linspace(0, 4, 5)        # 電流 5 点
A = np.array([0.0, 10.0, 20.0]) # 振幅 3 点
table = np.zeros((I.size, A.size))
print(table.shape)              # (5, 3) = (電流の数, 振幅の数)

for i in range(I.size):
    for a in range(A.size):
        table[i, a] = I[i] / 10 + A[a] / 100   # 仮の式
print(table)

print("A=10 の列（1本の IV 曲線）:", table[:, 1])
print("I=2 の行:", table[2, :])
print("縦方向（axis=0）の平均:", table.mean(axis=0))
print("横方向（axis=1）の平均:", table.mean(axis=1))

(5, 3)
[[0.  0.1 0.2]
 [0.1 0.2 0.3]
 [0.2 0.3 0.4]
 [0.3 0.4 0.5]
 [0.4 0.5 0.6]]
A=10 の列（1本の IV 曲線）: [0.1 0.2 0.3 0.4 0.5]
I=2 の行: [0.2 0.3 0.4]
縦方向（axis=0）の平均: [0.2 0.3 0.4]
横方向（axis=1）の平均: [0.1 0.2 0.3 0.4 0.5]


`table[:, 1]` の `:` は「その方向は全部」という意味です。**`V[:, a]` が振幅 A[a] のときの IV 曲線**になります。この書き方は第6章以降で何度も使います。

## 2.5 ブロードキャスト ―― for を使わずに表を作る

上の二重 `for` は、ブロードキャストを使うと1行で書けます。`I[:, None]` は形 `(5, 1)` の縦長、`A[None, :]` は形 `(1, 3)` の横長になり、足すと `(5, 3)` に自動で引き伸ばされます。

In [8]:
table2 = I[:, None] / 10 + A[None, :] / 100
print(table2.shape)
print(np.allclose(table, table2))   # 二重 for の結果と同じか

(5, 3)
True


研究の理論式 x = kA/Ω も同じ考え方で、「k を何通りか × A を何通りか」の表を一度に作れます。

In [9]:
k = np.array([0.05, 0.1, 0.2])
x1 = k[:, None] * A_list[None, :] / 0.5     # 形 (3, 33)
print(x1.shape)
print(np.round(x1[:, :4], 2))

(3, 33)
[[0.  0.2 0.4 0.6]
 [0.  0.4 0.8 1.2]
 [0.  0.8 1.6 2.4]]


## 2.6 ブールマスク ―― 条件に合う点だけ取り出す

`V > 0.2` のような比較は、True/False の配列を返します。`&`（かつ）・`|`（または）でつなげられます。かっこを忘れないこと。

In [10]:
V = np.array([0.0, 0.1, 0.25, 0.29, 0.30, 0.31, 0.5])
mask = (V > 0.26) & (V < 0.34)
print(mask)
print("窓の中の V:", V[mask])
print("その番号:", np.where(mask)[0])
print("窓の中に何点あるか:", mask.sum())   # True は 1 として数えられる

[False False False  True  True  True False]
窓の中の V: [0.29 0.3  0.31]
その番号: [3 4 5]
窓の中に何点あるか: 3


## 2.7 差分 np.diff

`np.diff(x)` は「隣どうしの差」を返します。長さは1つ減ります。IV 曲線の傾き dV/dI を求めるのに使います（第7章）。

In [11]:
I = np.array([0.0, 0.1, 0.2, 0.3, 0.4])
V = np.array([0.0, 0.0, 0.05, 0.05, 0.15])
print(np.diff(I))
print(np.diff(V) / np.diff(I))     # 傾き。0 のところは「平ら」＝ステップの候補

[0.1 0.1 0.1 0.1]
[0.  0.5 0.  1. ]


## 第2章の MATLAB ↔ Python 対応

| やりたいこと | MATLAB | Python |
|---|---|---|
| 0 の表を作る | `zeros(5, 3)` | `np.zeros((5, 3))`（形はタプルで渡す） |
| 形を調べる | `size(V)` | `V.shape` |
| 列を取り出す | `V(:, 2)` | `V[:, 1]` |
| 等間隔の点 | `linspace(0, 64, 33)` | `np.linspace(0, 64, 33)`（同じ） |
| 刻みで並べる | `0:0.25:1`（1 を含む） | `np.arange(0, 1, 0.25)`（1 を含まない） |
| 要素ごとの積 | `a .* b` | `a * b` |
| 行列の積 | `a * b` | `a @ b` |
| 縦方向の平均 | `mean(V, 1)` | `V.mean(axis=0)` |
| 条件に合う番号 | `find(x > 0)`（1 始まり） | `np.where(x > 0)[0]`（0 始まり） |
| 条件で取り出す | `x(x > 0)` | `x[x > 0]`（同じ） |
| 一部を代入して別にする | `B = A(1:3)`（コピーになる） | `B = A[0:3].copy()`（`copy` しないとビュー） |
| 縦長・横長にする | `I(:)`、`A(:)'` | `I[:, None]`、`A[None, :]` |

全章の分をまとめたものは [`対応表.md`](../対応表.md) にあります。

## 穴埋め練習

2.5 のブロードキャストを使って、仮の IV マップ（電流 9 点 × 振幅 3 点）を作り、A = 20 の列を取り出します。

In [12]:
# 穴埋め：___ を埋めてから、全部の行を選んで Cmd + /（Windows は Ctrl + /）を押し、行頭の # を外して実行する
# I = np.linspace(0, 4, ___)                   # 9 点
# A = np.array([0.0, 10.0, 20.0])
# T = I[:, ___] / 10 + A[___, :] / 100
# print(T.shape)                               # (9, 3) になるはず
# print(T[:, ___])                             # A = 20 の列

<details><summary>穴埋めの答え（クリックで開く）</summary>

上から `9`、`None`、`None`、`2` です。A = 20 は `A` の3番目なので、0 から数えて `2` です。
</details>

下のセルは答えを埋めた版です。自分で埋めた結果と同じになるか確かめてください。

In [13]:
# 答え合わせ（埋めた版）
I = np.linspace(0, 4, 9)
A = np.array([0.0, 10.0, 20.0])
T = I[:, None] / 10 + A[None, :] / 100
print(T.shape)
print(T[:, 2])

(9, 3)
[0.2  0.25 0.3  0.35 0.4  0.45 0.5  0.55 0.6 ]


## 予想してから実行

**問題：** 次の3つは、それぞれ何と表示されるでしょうか。
1. `x = np.arange(6)` のあと `y = x[2:4]` とし、`y[:] = 0` としたときの `x`
2. `np.arange(0, 1, 0.25).size`
3. `(np.zeros((3, 1)) + np.zeros((1, 4))).shape`

**先に予想を紙かメモに書いてから**、次のセルを実行してください。

In [14]:
x = np.arange(6)
y = x[2:4]
y[:] = 0
print(x)
print(np.arange(0, 1, 0.25).size)
print((np.zeros((3, 1)) + np.zeros((1, 4))).shape)

[0 1 0 0 4 5]
4
(3, 4)


<details><summary>解説（実行してから開く）</summary>

1. `[0 1 0 0 4 5]`。`y` は `x` の一部をのぞいている**ビュー**なので、`y` を書き換えると `x` も変わります（2.3）。MATLAB ならコピーになるので `x` は変わりません。
2. `4`（0, 0.25, 0.5, 0.75）。`arange` は終わりの 1 を含みません。MATLAB の `0:0.25:1` は 5 個です。
3. `(3, 4)`。縦長 (3, 1) と横長 (1, 4) が、ブロードキャストで (3, 4) に引き伸ばされます。
</details>

## 練習問題

1. `A_list` のうち 20 以上 40 以下のものだけを取り出してください。何点ありますか。
2. Ω₁ = 0.5, Ω₂ = 0.8 として、n = -3〜3、m = -3〜3 の全部の組の電圧 V = nΩ₁ + mΩ₂ を、ブロードキャストで形 (7, 7) の表にしてください。その中で 0 < V < 1.5 のものはいくつありますか。

<details><summary>答え</summary>

```python
# 1.
sel = A_list[(A_list >= 20) & (A_list <= 40)]
print(sel, sel.size)     # 11 点
# 2.
nn = np.arange(-3, 4); mm = np.arange(-3, 4)
VV = nn[:, None] * 0.5 + mm[None, :] * 0.8
print(VV.shape, ((VV > 0) & (VV < 1.5)).sum())
```
</details>

In [15]:
sel = A_list[(A_list >= 20) & (A_list <= 40)]
print(sel, sel.size)
nn = np.arange(-3, 4); mm = np.arange(-3, 4)
VV = nn[:, None] * 0.5 + mm[None, :] * 0.8
print(VV.shape, ((VV > 0) & (VV < 1.5)).sum())

[20. 22. 24. 26. 28. 30. 32. 34. 36. 38. 40.] 11
(7, 7) 12


## まとめ
- NumPy の `*` `/` は要素ごと。配列にまとめて計算すると速い（ベクトル化）。
- IV マップは `V[電流, 振幅]` の2次元配列。`V[:, a]` が1本の IV 曲線。
- スライスはビュー。独立させたいときは `.copy()`。
- ブロードキャストとブールマスクで、`for` を減らせる。